# 05 — Experiment C: GRU "Neural ELO" Sequence Model

A fourth experiment alongside Exp 0 (ELO baseline), Exp A (ELO-free XGBoost), Exp B (ELO-included
XGBoost) from `03_model_training.ipynb` / `04_model_evaluation.ipynb`.

**The idea:** instead of hand-crafted point-in-time summary features + a static classifier, a
GRU consumes each player's raw chronological sequence of past matches and learns an evolving
skill/form *state*. Conceptually the neural counterpart to Exp A: can a learned sequence model
reconstruct (or beat) ELO using only the win/loss graph and match context — **no `EloBefore` /
`EloAfter` / `elo_diff` anywhere in the model input**?

**Expectation set going in (honest prior):** with ~17k matches and short per-player
sequences, matching Exp A/B would already be a good result; dramatically beating ELO seemed
unlikely (data-limited). A null/tie was explicitly called out as a legitimate, reportable
finding. **Spoiler, backed by a leakage audit and a 3-seed stability check below: this is NOT
what happened** — see the written verdict at the end. **Note added after this notebook was written (see `06_confidence_intervals_cv.ipynb`, `07_xgboost_trajectory.ipynb`, `08_gru_elo.ipynb`, `09_walkforward_top_models.ipynb`): the single-split headline below (Exp C beats everything) does NOT survive statistical and temporal validation unchanged — read the Written Verdict's closing "Where this stands today" section, added after those notebooks were run.**

**Reuses, unchanged, from the feature-engineering notebook:**
- The exact same `matches_export.csv` raw data and the exact same time-based `Split` marker
  (`features.parquet`'s `Split` column — train: 2017-05-08 .. 2025-01-18 / test: 2025-01-19 ..
  2026-07-08, 3,426 test matches / 6,852 duplicate-and-flip rows).
- The exact same **duplicate-and-flip symmetric expansion** (`PlayerId`/`OpponentId`/`Label`
  from `features.parquet`) — this is also this notebook's answer to the symmetry requirement:
  rather than an architecturally-antisymmetric head, **P(A beats B) = 1 − P(B beats A) is
  guaranteed by construction** because every match contributes two rows with swapped
  player/opponent roles and swapped labels, exactly as in `02_feature_engineering.ipynb`'s
  duplicate-and-flip step.
- The exact same TRAIN-only chronological carve fractions as `03_model_training.ipynb`
  (`calib_frac=0.15`, `early_stop_frac=0.10` of the remainder) so the four experiments are
  compared under identical data-splitting discipline.
- The exact same evaluation metrics (accuracy, Brier — primary, log-loss, AUC, calibration
  curve) on the identical 6,852-row held-out TEST set.

In [ ]:
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.use_deterministic_algorithms(False)  # GRU has no deterministic CPU kernel path for this op set

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cpu')
print(f'PyTorch {torch.__version__}, device={device}, threads={torch.get_num_threads()}')

## Step 1 — Load raw export + the P2 engineered frame (for identical Split/Label/symmetry)

`features.parquet` is loaded ONLY for its `PlayerId`/`OpponentId`/`Label`/`Split`/`Datum`/
`SusretId` columns — i.e. the exact duplicate-and-flip row set and time-based split boundary
already established in Phase P2/P3. All of the GRU's own per-step sequence features are
(re)derived from the raw `matches_export.csv` below, independently of the Exp A/B tabular
feature engineering.

In [ ]:
raw = pd.read_csv(DATA_DIR / 'matches_export.csv', parse_dates=['Datum'])
raw['Datum'] = pd.to_datetime(raw['Datum']).dt.tz_localize(None)
raw = raw.sort_values(['Datum', 'SusretId']).reset_index(drop=True)

sym = pd.read_parquet(DATA_DIR / 'features.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)

print(f'raw matches: {raw.shape}')
print(f'sym (duplicate-and-flip) frame: {sym.shape}')
print(f"Split date (from P2): {TAGS['split_date']}, train matches: {TAGS['n_matches_train']:,}, "
      f"test matches: {TAGS['n_matches_test']:,}")
sym[['SusretId', 'Datum', 'PlayerId', 'OpponentId', 'Label', 'Perspective', 'Split']].head()

## Step 2 — Long-format player-match log (same construction as `02_feature_engineering.ipynb` Step 1)

One row per (player, match). `Won` is the outcome of THAT historical match — this is the raw
per-step signal a hand-crafted "career win-rate" feature would otherwise summarize into a single
number; the GRU gets to see the actual sequence instead.

In [ ]:
common_cols = ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'LigaTip', 'LigaKoeficjent', 'Lokacija']

winner_rows = raw[common_cols + ['WinnerId', 'LoserId', 'W_Set1', 'W_Set2', 'W_Set3', 'L_Set1', 'L_Set2', 'L_Set3']].rename(
    columns={'WinnerId': 'IgracId', 'LoserId': 'OpponentId',
             'W_Set1': 'MySet1', 'W_Set2': 'MySet2', 'W_Set3': 'MySet3',
             'L_Set1': 'OppSet1', 'L_Set2': 'OppSet2', 'L_Set3': 'OppSet3'}
).assign(Won=1)

loser_rows = raw[common_cols + ['LoserId', 'WinnerId', 'L_Set1', 'L_Set2', 'L_Set3', 'W_Set1', 'W_Set2', 'W_Set3']].rename(
    columns={'LoserId': 'IgracId', 'WinnerId': 'OpponentId',
             'L_Set1': 'MySet1', 'L_Set2': 'MySet2', 'L_Set3': 'MySet3',
             'W_Set1': 'OppSet1', 'W_Set2': 'OppSet2', 'W_Set3': 'OppSet3'}
).assign(Won=0)

player_log = pd.concat([winner_rows, loser_rows], ignore_index=True)
player_log['LokacijaNorm'] = player_log['Lokacija'].str.strip().str.lower()
# CRITICAL: strict ascending (IgracId, Datum, SusretId) order -- every point-in-time feature
# below only ever looks at rows ABOVE the current one within a player's own group, i.e.
# strictly BEFORE this match. Same discipline as notebook 02 / the SQL ROW_NUMBER pattern.
player_log = player_log.sort_values(['IgracId', 'Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
print(f'player_log: {player_log.shape}')
player_log.head(3)

## Step 3 — Per-step point-in-time features

For every historical match, compute (all strictly as of that match, no different from Exp A/B's
own point-in-time discipline):
- `Won` — outcome
- `SetMargin` — total games margin (missing 3rd set treated as 0-0, i.e. contributes nothing)
- `DaysSinceLastMatch` — rest before that match (`NaN` on a player's own debut match)
- `CareerWinRateBefore` — win/loss-graph-only rolling career win rate BEFORE that match (needed
  below to compute the *opponent's* quality at that time — never touches `EloBefore`/`EloAfter`)
- `HomeShareBefore` — fraction of the player's matches before that match played at that same venue

Cross-checked against the SQL-exported `MatchCountBefore` exactly as in notebook 02 — this is the
same leakage-audit discipline, reapplied here because this notebook rebuilds the player-log
independently rather than reusing `features.parquet`'s tabular diff columns.

In [ ]:
g = player_log.groupby('IgracId', sort=False)
player_log['MatchCountBefore'] = g.cumcount()
player_log['HomeCountBefore'] = player_log.groupby(['IgracId', 'LokacijaNorm'], sort=False).cumcount()
player_log['HomeShareBefore'] = np.where(
    player_log['MatchCountBefore'] > 0,
    player_log['HomeCountBefore'] / player_log['MatchCountBefore'],
    np.nan,
)
player_log['DaysSinceLastMatch'] = player_log.groupby('IgracId')['Datum'].diff().dt.days
# ELO-free: built exclusively from the win/loss graph, never from EloBefore/EloAfter --
# same SQL-comment discipline as Part 1.5's CareerWinRateBefore.
player_log['CareerWinRateBefore'] = g['Won'].transform(lambda s: s.shift().expanding().mean())

for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']:
    player_log[c + '_f'] = player_log[c].fillna(0)
player_log['SetMargin'] = (
    (player_log['MySet1_f'] + player_log['MySet2_f'] + player_log['MySet3_f'])
    - (player_log['OppSet1_f'] + player_log['OppSet2_f'] + player_log['OppSet3_f'])
)
player_log = player_log.drop(columns=[c + '_f' for c in ['MySet1', 'MySet2', 'MySet3', 'OppSet1', 'OppSet2', 'OppSet3']])

# Cross-check vs the SQL-exported point-in-time match count (same discipline as notebook 02 Step 2)
check_w = raw[['SusretId', 'WinnerId', 'WinnerMatchCountBefore']].rename(columns={'WinnerId': 'IgracId', 'WinnerMatchCountBefore': 'Exp'})
check_l = raw[['SusretId', 'LoserId', 'LoserMatchCountBefore']].rename(columns={'LoserId': 'IgracId', 'LoserMatchCountBefore': 'Exp'})
chk = pd.concat([check_w, check_l])
mm = player_log.merge(chk, on=['SusretId', 'IgracId'], how='left')
mismatches = (mm['MatchCountBefore'] != mm['Exp']).sum()
assert mismatches == 0, 'pandas point-in-time ordering does not match the SQL export!'
print(f'MatchCountBefore cross-check vs SQL export: {mismatches} mismatches -- OK.')
player_log[['Won', 'SetMargin', 'DaysSinceLastMatch', 'LigaKoeficjent', 'HomeShareBefore', 'CareerWinRateBefore']].describe()

## Step 4 — Opponent-quality-at-that-time proxy (the "who did I beat" signal)

By design: each history step must carry *who was beaten*, not just win/loss. Rather than
recursively embedding the opponent's own history (out of scope — that's GNN territory), we attach
a single scalar: **the opponent's own `CareerWinRateBefore` as of that same historical match** —
reusing the exact point-in-time column already engineered above, looked up via a self-merge on
`(SusretId, OpponentId)`. This is ELO-free by construction (built only from `Won`) and gives the
GRU a cheap, point-in-time-safe opponent-strength signal for every step in the sequence.

In [ ]:
opp_cwr = player_log[['SusretId', 'IgracId', 'CareerWinRateBefore']].rename(
    columns={'IgracId': 'OpponentId', 'CareerWinRateBefore': 'OpponentCareerWinRateBefore'})
player_log = player_log.merge(opp_cwr, on=['SusretId', 'OpponentId'], how='left', validate='many_to_one')
print(f"OpponentCareerWinRateBefore NaN share (opponent's own debut match): "
      f"{player_log['OpponentCareerWinRateBefore'].isna().mean():.2%}")
player_log[['OpponentCareerWinRateBefore']].describe()

## Step 5 — Build the per-step feature array + history precomputation

Sequence length capped at `MAX_LEN=24` (within the 20–30 spec range). Each real step is a
7-dim vector: `[Won, SetMarginScaled, DaysRestScaled, LigaKoeficjentScaled, HomeShareBefore,
OpponentCareerWinRateBefore, IsPlaceholder]`.

**Zero-history (debut match) handling:** a `GRU` cannot consume a zero-length sequence. Rather
than special-casing it in the model, a single learned-signal **placeholder step**
(`IsPlaceholder=1`, neutral `0.5` opponent-quality, everything else `0`) is substituted — the
network can learn what "no history yet" means from this flag, same spirit as the `elo_confidence`
meta-feature in Exp A/B (an explicit "how much do I actually know about this player" signal).

**History extraction is O(1) per lookup:** since `player_log` is sorted `(IgracId, Datum,
SusretId)`, each `IgracId`'s rows are contiguous, so the strictly-prior history for row `gpos` is
just `step_feats[group_start : gpos]` (endpoint EXCLUDES the row itself — this is the entire
leakage guarantee, verified explicitly below).

**Train/test hygiene note (adversarial audit, 2026-07-10):** `LIGAKOEF_SCALE` below is computed from TRAIN ONLY, not the full frame -- see the code comment and Step 15 for the before/after verification that this fix is numerically inert.

In [ ]:
MAX_LEN = 24
N_FEATS = 7  # Won, SetMarginScaled, DaysRestScaled, LigaKoefScaled, HomeShareBefore, OppCareerWinRateBefore, IsPlaceholder

SETMARGIN_SCALE = 12.0   # ~max plausible games-margin per set * a couple sets -- fixed domain constant, not data-derived
DAYSREST_CLIP = 90.0     # a fixed domain constant matching this project's usual 90-day rest-cap convention, not data-derived

# --- Train/test hygiene fix (adversarial audit, 2026-07-10) ---
# LIGAKOEF_SCALE was originally hardcoded to 8.0, described as "observed max LigaKoeficjent (see
# describe() above)" -- but that describe() (Step 3 output) was computed over `player_log` BEFORE
# any train/test split existed, i.e. over the FULL train+test frame. LigaKoeficjent's max is only
# 7 within TRAIN; 8 appears for the first time only in the TEST period (a new razina level
# introduced after the split date). Using a TEST-period fact to set a scaling constant is a
# train/test hygiene violation, even though it never touches TEST labels directly. Recomputed here
# from TRAIN ONLY. (Robustness-checked with a real before/after retrain in Step 15 below --
# expected to be numerically inert, since a GRU's first-layer weights can absorb any fixed linear
# rescale of an input dimension.)
_split_date = pd.Timestamp(TAGS['split_date'])
_train_mask_pl = player_log['Datum'] < _split_date
LIGAKOEF_SCALE = float(player_log.loc[_train_mask_pl, 'LigaKoeficjent'].max())  # train-only max
print(f"LIGAKOEF_SCALE (train-only) = {LIGAKOEF_SCALE}  "
      f"(overall max = {player_log['LigaKoeficjent'].max()}, "
      f"test-only max = {player_log.loc[~_train_mask_pl, 'LigaKoeficjent'].max()} "
      f"-- confirms a razina level unseen in TRAIN appears only in TEST)")

step_feats = np.zeros((len(player_log), N_FEATS), dtype=np.float32)
step_feats[:, 0] = player_log['Won'].values
step_feats[:, 1] = np.clip(player_log['SetMargin'].values / SETMARGIN_SCALE, -3, 3)
days_rest = player_log['DaysSinceLastMatch'].fillna(DAYSREST_CLIP).clip(upper=DAYSREST_CLIP).values
step_feats[:, 2] = days_rest / DAYSREST_CLIP
step_feats[:, 3] = player_log['LigaKoeficjent'].values / LIGAKOEF_SCALE
step_feats[:, 4] = player_log['HomeShareBefore'].fillna(0.0).values
step_feats[:, 5] = player_log['OpponentCareerWinRateBefore'].fillna(0.5).values
step_feats[:, 6] = 0.0  # real step

PLACEHOLDER_VEC = np.zeros(N_FEATS, dtype=np.float32)
PLACEHOLDER_VEC[5] = 0.5   # neutral opponent quality
PLACEHOLDER_VEC[6] = 1.0   # IsPlaceholder flag

player_log['_gpos'] = np.arange(len(player_log))
first_idx = player_log.groupby('IgracId')['_gpos'].min().to_dict()
idx_lookup = dict(zip(zip(player_log['IgracId'], player_log['SusretId']), player_log['_gpos']))

def build_history(igrac_id, susret_id):
    """(seq[MAX_LEN, N_FEATS] float32, length) -- history STRICTLY before this match."""
    gpos = idx_lookup[(igrac_id, susret_id)]
    start = first_idx[igrac_id]
    p = gpos - start  # == this player's MatchCountBefore at this match, cross-checked above
    seq = np.zeros((MAX_LEN, N_FEATS), dtype=np.float32)
    if p == 0:
        seq[0] = PLACEHOLDER_VEC
        length = 1
    else:
        lo = max(start, gpos - MAX_LEN)
        chunk = step_feats[lo:gpos]  # excludes gpos itself -> strictly prior
        length = chunk.shape[0]
        seq[:length] = chunk
    return seq, length

n_rows = len(player_log)
HIST_ARRAY = np.zeros((n_rows, MAX_LEN, N_FEATS), dtype=np.float32)
LEN_ARRAY = np.zeros(n_rows, dtype=np.int64)
igrac_arr, susret_arr = player_log['IgracId'].values, player_log['SusretId'].values
for i in range(n_rows):
    HIST_ARRAY[i], LEN_ARRAY[i] = build_history(igrac_arr[i], susret_arr[i])

print(f'Precomputed histories for {n_rows:,} player-match rows.')
print(f'Sequence length -- min={LEN_ARRAY.min()}, median={np.median(LEN_ARRAY):.0f}, max={LEN_ARRAY.max()}')
print(f'Debut (placeholder) rows: {(player_log["MatchCountBefore"] == 0).sum():,}')

### Leakage spot-check (mandatory — Part 2.2/2.2.1 discipline, reapplied here)

1. `p` (position derived from the global sort) exactly equals the already-cross-checked
   `MatchCountBefore` for every row — confirms group contiguity assumption holds everywhere, not
   just on average.
2. A concrete spot-check on a real TEST-period player: the history window's `Datum` values must
   all be strictly before the predicted match's own `Datum` — checked directly, not assumed.

In [ ]:
p_check = (player_log['_gpos'] - player_log['IgracId'].map(first_idx))
assert (p_check == player_log['MatchCountBefore']).all(), 'group contiguity assumption violated!'
print('OK -- p (derived from global sort) == MatchCountBefore for all rows (group contiguity holds).')

test_period_start = pd.Timestamp(TAGS['split_date'])
sample = player_log.loc[player_log['Datum'] >= test_period_start].iloc[100]
gpos, start = sample['_gpos'], first_idx[sample['IgracId']]
hist_rows = player_log.iloc[start:gpos]
print(f"\nSpot check -- IgracId={sample['IgracId']}, SusretId={sample['SusretId']}, "
      f"Datum={sample['Datum'].date()}, MatchCountBefore={sample['MatchCountBefore']}")
print(f'History rows found: {len(hist_rows)} (expected {sample["MatchCountBefore"]})')
print(f"Max Datum in history: {hist_rows['Datum'].max().date()} -- must be < {sample['Datum'].date()}")
assert (hist_rows['Datum'] < sample['Datum']).all(), 'LEAKAGE: a history row is not strictly prior!'
print('OK -- every history row is strictly before the predicted match date. No leakage.')

## Step 6 — Resolve `features.parquet` rows onto history-array positions

Every `(PlayerId, SusretId)` / `(OpponentId, SusretId)` pair in `sym` (the duplicate-and-flip
frame) corresponds to exactly one `player_log` row — reuse `idx_lookup` to fetch the precomputed
history for both towers of every row, for the SAME `Split`/`Label`/row-set as Exp A/B.

In [ ]:
sym = sym.sort_values(['Datum', 'SusretId', 'Perspective'], kind='mergesort').reset_index(drop=True)
player_pos = np.array([idx_lookup[(pid, sid)] for pid, sid in zip(sym['PlayerId'], sym['SusretId'])])
opp_pos = np.array([idx_lookup[(oid, sid)] for oid, sid in zip(sym['OpponentId'], sym['SusretId'])])
print(f'Resolved player/opponent history positions for {len(sym):,} rows.')
print(sym['Split'].value_counts())

## Step 7 — Chronological carve within TRAIN (identical fractions to `03_model_training.ipynb`)

`core` (fit) / `eval` (early-stopping monitor) / `calib` (post-hoc calibration) — same
`calib_frac=0.15`, `early_stop_frac=0.10` of the remainder, carved at MATCH level so both
perspective rows of a match always land in the same bucket. TEST is never touched until scoring.

In [ ]:
train_matches = (
    sym.loc[sym['Split'] == 'train', ['SusretId', 'Datum']]
    .drop_duplicates()
    .sort_values(['Datum', 'SusretId'], kind='mergesort')
    .reset_index(drop=True)
)
n_train_matches = len(train_matches)
CALIB_FRAC, EARLY_STOP_FRAC = 0.15, 0.10

n_calib = int(round(n_train_matches * CALIB_FRAC))
fit_order = train_matches.iloc[: n_train_matches - n_calib].reset_index(drop=True)
calib_ids = set(train_matches.iloc[n_train_matches - n_calib:]['SusretId'])

n_fit = len(fit_order)
n_eval = int(round(n_fit * EARLY_STOP_FRAC))
core_ids = set(fit_order.iloc[: n_fit - n_eval]['SusretId'])
eval_ids = set(fit_order.iloc[n_fit - n_eval:]['SusretId'])

sym['Bucket'] = np.where(sym['Split'] == 'test', 'test',
                  np.where(sym['SusretId'].isin(calib_ids), 'calib',
                  np.where(sym['SusretId'].isin(eval_ids), 'eval', 'core')))
print(sym['Bucket'].value_counts())
assert sym.loc[sym['Bucket'] == 'test'].shape[0] == TAGS['n_matches_test'] * 2

## Step 8 — Two-tower shared-weight GRU model

- **Shared GRU** (`hidden_size=24`, within the 16–32 spec range) encodes each player's
  chronological sequence into a final hidden state = a learned skill/form embedding at that
  point in time. Same weights for both towers (true siamese network).
- **Combine:** `[h_player, h_opponent, h_player − h_opponent]` (concat AND difference, per spec)
  → small MLP head (`Linear(3H→16) → ReLU → Dropout → Linear(16→1)`) → logit (sigmoid applied at
  inference/eval, `BCEWithLogitsLoss` used for training).
- **Symmetry:** NOT an antisymmetric head — see the notebook intro. Symmetry is guaranteed by the
  duplicate-and-flip dataset (each match contributes a `winner_pov` row AND a `loser_pov` row with
  swapped player/opponent identity and swapped label), exactly as Exp A/B's tabular pipeline.
- **Anti-overfit:** dropout (`0.3`) on both the GRU output and inside the MLP head, weight decay
  (`1e-4`, AdamW), a capped sequence length (`MAX_LEN=24`), and early stopping on the chronological
  `eval` bucket's Brier score (never TEST, never the calibration slice).
- Fixed seed (`RANDOM_STATE=42`, matching the XGBoost experiments' convention) — but NN training
  retains residual stochasticity; a 3-seed stability check is reported in the written verdict.

In [ ]:
HIST_T = torch.from_numpy(HIST_ARRAY)
LEN_T = torch.from_numpy(LEN_ARRAY)


class TwoTowerGRU(nn.Module):
    def __init__(self, input_size=N_FEATS, hidden_size=24, dropout=0.3, mlp_hidden=16):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o], dim=1)
        return self.head(combined).squeeze(-1)


def bucket_tensors(bucket_name):
    mask = (sym['Bucket'] == bucket_name).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
    return HIST_T[pp], LEN_T[pp], HIST_T[op], LEN_T[op], y

core_pseq, core_plen, core_oseq, core_olen, core_y = bucket_tensors('core')
eval_pseq, eval_plen, eval_oseq, eval_olen, eval_y = bucket_tensors('eval')
calib_pseq, calib_plen, calib_oseq, calib_olen, calib_y = bucket_tensors('calib')
test_pseq, test_plen, test_oseq, test_olen, test_y = bucket_tensors('test')
print(f'core={core_y.shape[0]:,}  eval={eval_y.shape[0]:,}  calib={calib_y.shape[0]:,}  test={test_y.shape[0]:,}')

## Step 9 — Training loop (manual full-tensor mini-batches, early stopping on `eval` Brier)

Batches are built via direct tensor-index slicing rather than a `torch.utils.data.DataLoader` —
with only ~21k core rows this comfortably fits in memory, and avoids per-sample Python
`__getitem__` overhead (measured ~25x slower on this CPU for no benefit at this data size).

In [ ]:
HIDDEN_SIZE, DROPOUT, WEIGHT_DECAY, LR = 24, 0.3, 1e-4, 2e-3
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 512, 60, 8

torch.manual_seed(RANDOM_STATE)
model = TwoTowerGRU(hidden_size=HIDDEN_SIZE, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
n_core = core_y.shape[0]

def eval_bucket(p_seq, p_len, o_seq, o_len, y):
    model.eval()
    with torch.no_grad():
        logit = model(p_seq, p_len, o_seq, o_len)
    return brier_score_loss(y.numpy(), torch.sigmoid(logit).numpy()), logit

best_brier, best_state, epochs_no_improve = np.inf, None, 0
history, rng = [], np.random.RandomState(RANDOM_STATE)

t0 = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    perm = rng.permutation(n_core)
    total_loss, n_batches = 0.0, 0
    for start in range(0, n_core, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        optimizer.zero_grad()
        logit = model(core_pseq[idx], core_plen[idx], core_oseq[idx], core_olen[idx])
        loss = criterion(logit, core_y[idx])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    val_brier, _ = eval_bucket(eval_pseq, eval_plen, eval_oseq, eval_olen, eval_y)
    history.append({'epoch': epoch, 'train_loss': total_loss / n_batches, 'val_brier': val_brier})

    improved = val_brier < best_brier - 1e-5
    if improved:
        best_brier, best_state, epochs_no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        epochs_no_improve += 1

    if epoch % 5 == 0 or improved or epoch == 1:
        print(f'epoch {epoch:3d}  train_loss={total_loss/n_batches:.4f}  val_brier={val_brier:.5f}'
              f"{'  <- best' if improved else ''}")
    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (no improvement in {PATIENCE} epochs).')
        break

model.load_state_dict(best_state)
print(f'\nTraining done in {time.time()-t0:.1f}s. Best eval-bucket Brier = {best_brier:.5f}')
history_df = pd.DataFrame(history)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df['epoch'], history_df['train_loss'], label='train loss (BCE, core)', color='#2563eb')
ax2 = ax.twinx()
ax2.plot(history_df['epoch'], history_df['val_brier'], label='val Brier (eval bucket)', color='#dc2626')
ax.set_xlabel('epoch'); ax.set_ylabel('train BCE loss', color='#2563eb'); ax2.set_ylabel('val Brier', color='#dc2626')
ax.set_title('Exp C training curve -- early stopping on chronological eval-bucket Brier')
fig.tight_layout()
plt.savefig('../models/exp_c_training_curve.png', dpi=110)
plt.show()

## Step 10 — Calibration on the `calib` bucket (never used for fitting or early stopping)

Two candidates fit on the SAME calibration slice, by design ("temperature scaling or
isotonic"): **temperature scaling** (single scalar `T` minimizing BCE on calibration logits — the
standard, lightweight NN calibration method) and **isotonic regression** on the raw sigmoid
probabilities. Whichever gives the lower Brier ON THE CALIBRATION SLICE ITSELF is applied to TEST
— same "pick the empirically better calibrator" discipline used for Exp A/B's isotonic-vs-sigmoid
choice in `03_model_training.ipynb`.

In [ ]:
model.eval()
with torch.no_grad():
    calib_logit = model(calib_pseq, calib_plen, calib_oseq, calib_olen)
    test_logit = model(test_pseq, test_plen, test_oseq, test_olen)

calib_logit_np, calib_y_np = calib_logit.numpy(), calib_y.numpy()
test_logit_np, test_y_np = test_logit.numpy(), test_y.numpy()

# --- Temperature scaling: 1 scalar T minimizing BCE on the calibration slice ---
T_param = torch.nn.Parameter(torch.ones(1))
calib_logit_t, calib_y_t = torch.from_numpy(calib_logit_np), torch.from_numpy(calib_y_np)
opt_T = torch.optim.LBFGS([T_param], lr=0.05, max_iter=100)

def _closure():
    opt_T.zero_grad()
    loss = nn.functional.binary_cross_entropy_with_logits(calib_logit_t / T_param, calib_y_t)
    loss.backward()
    return loss

opt_T.step(_closure)
T_value = float(T_param.detach().clamp(min=1e-2))

p_calib_raw = 1 / (1 + np.exp(-calib_logit_np))
p_calib_temp = torch.sigmoid(calib_logit_t / T_value).numpy()

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(p_calib_raw, calib_y_np)
p_calib_iso = iso.predict(p_calib_raw)

brier_raw = brier_score_loss(calib_y_np, p_calib_raw)
brier_temp = brier_score_loss(calib_y_np, p_calib_temp)
brier_iso = brier_score_loss(calib_y_np, p_calib_iso)
print(f'Fitted temperature T = {T_value:.4f}')
print(f'Calib-slice Brier -- raw: {brier_raw:.5f}, temperature: {brier_temp:.5f}, isotonic: {brier_iso:.5f}')

p_test_raw = 1 / (1 + np.exp(-test_logit_np))
if brier_iso < brier_temp:
    calib_method = 'isotonic'
    p_test_calibrated = iso.predict(p_test_raw)
else:
    calib_method = 'temperature'
    p_test_calibrated = torch.sigmoid(torch.from_numpy(test_logit_np) / T_value).numpy()
print(f'Chosen calibration method for TEST: {calib_method}')

## Step 11 — Score TEST (the identical 6,852-row set as Exp 0/A/B) — 4-way comparison

In [ ]:
def compute_metrics(y_true, p, label):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        'experiment': label,
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
        'auc': roc_auc_score(y_true, p),
    }

preds = pd.read_parquet(DATA_DIR / 'test_predictions.parquet')

# Independent alignment check: the GRU test bucket must be the EXACT same (SusretId, PlayerId,
# OpponentId, Label) row set as Exp 0/A/B's test_predictions.parquet -- not just the same size.
test_bucket_keys = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'PlayerId', 'OpponentId', 'Label']]
merge_check = test_bucket_keys.merge(preds[['SusretId', 'PlayerId', 'OpponentId', 'Label']],
                                      on=['SusretId', 'PlayerId', 'OpponentId'], suffixes=('_gru', '_exp'))
assert len(merge_check) == len(test_bucket_keys) == len(preds), 'Row-set mismatch vs Exp A/B test set!'
assert (merge_check['Label_gru'] == merge_check['Label_exp']).all(), 'Label mismatch vs Exp A/B test set!'
print(f'OK -- Exp C test bucket is the exact same {len(preds):,}-row set as Exp 0/A/B, labels match.')

elo_brier = brier_score_loss(preds['Label'], preds['p_exp0_elo'])
comparison = pd.DataFrame([
    compute_metrics(preds['Label'].values, preds['p_exp0_elo'].values, 'Exp 0 - ELO baseline (no ML)'),
    compute_metrics(preds['Label'].values, preds['p_expA_calibrated'].values, 'Exp A - ELO-free XGBoost (calibrated)'),
    compute_metrics(preds['Label'].values, preds['p_expB_calibrated'].values, 'Exp B - ELO-included XGBoost (calibrated)'),
    compute_metrics(test_y_np, p_test_calibrated, 'Exp C - ELO-free GRU (calibrated)'),
]).set_index('experiment')
comparison['brier_delta_vs_elo_pct'] = (comparison['brier'] - elo_brier) / elo_brier * 100
comparison_display = comparison.round({'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})
comparison_display

In [ ]:
print('Headline verdict:')
for exp in ['Exp A - ELO-free XGBoost (calibrated)', 'Exp B - ELO-included XGBoost (calibrated)',
            'Exp C - ELO-free GRU (calibrated)']:
    d = comparison.loc[exp, 'brier_delta_vs_elo_pct']
    print(f"  {exp}: Brier {d:+.2f}% vs ELO baseline ({'WORSE' if d > 0 else 'better'} -- higher Brier = worse).")

# Raw (uncalibrated) GRU, for the same calibration-effect sanity check as notebook 03
raw_metrics = compute_metrics(test_y_np, p_test_raw, 'Exp C - raw (uncalibrated)')
cal_metrics = compute_metrics(test_y_np, p_test_calibrated, 'Exp C - calibrated')
pd.DataFrame([raw_metrics, cal_metrics]).set_index('experiment').round(5)

## Step 12 — Overlaid calibration curves, Exp 0 / A / B / C

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
curves = [
    ('p_exp0_elo', 'Exp 0 - ELO formula', preds, dict(marker='s', linestyle='--')),
    ('p_expA_calibrated', 'Exp A - ELO-free XGB', preds, dict(marker='o', linestyle='-')),
    ('p_expB_calibrated', 'Exp B - ELO-included XGB', preds, dict(marker='^', linestyle='-')),
]
for col, label, frame, style in curves:
    frac_pos, mean_pred = calibration_curve(frame['Label'].values, frame[col].values, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, frac_pos, label=label, **style)

frac_pos_c, mean_pred_c = calibration_curve(test_y_np, p_test_calibrated, n_bins=10, strategy='uniform')
ax.plot(mean_pred_c, frac_pos_c, label='Exp C - ELO-free GRU', marker='D', linestyle='-', color='#7c3aed')

ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed win rate')
ax.set_title('Calibration (Reliability Diagram) -- Exp 0 vs A vs B vs C, TEST set')
ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../models/calibration_curve_4way.png', dpi=120)
plt.show()

## Step 13 — Permutation-style importance for the GRU (Exp A/B tree-importance analog)

XGBoost has native gain-based feature importance; a GRU does not. The model-agnostic equivalent:
shuffle ONE of the 6 real per-step features (across all rows, for both towers simultaneously,
holding each row's own sequence length/padding intact) and measure TEST Brier degradation. Larger
degradation = the model relies on that feature more.

In [ ]:
FEAT_NAMES = ['Won', 'SetMarginScaled', 'DaysRestScaled', 'LigaKoefScaled', 'HomeShareBefore', 'OpponentCareerWinRateBefore']

def permutation_importance_gru(feat_idx, n_repeats=3, seed=42):
    rng_local = np.random.RandomState(seed)
    baseline_p = p_test_calibrated
    baseline_brier = brier_score_loss(test_y_np, baseline_p)
    degradations = []
    for _ in range(n_repeats):
        perm_idx = rng_local.permutation(test_pseq.shape[0])
        p_seq_perm, o_seq_perm = test_pseq.clone(), test_oseq.clone()
        p_seq_perm[:, :, feat_idx] = test_pseq[perm_idx, :, feat_idx]
        o_seq_perm[:, :, feat_idx] = test_oseq[perm_idx, :, feat_idx]
        with torch.no_grad():
            logit_perm = model(p_seq_perm, test_plen, o_seq_perm, test_olen)
        p_perm_raw = torch.sigmoid(logit_perm).numpy()
        p_perm = iso.predict(p_perm_raw) if calib_method == 'isotonic' else \
            torch.sigmoid(logit_perm / T_value).numpy()
        degradations.append(brier_score_loss(test_y_np, p_perm) - baseline_brier)
    return np.mean(degradations)

perm_importance = pd.Series({name: permutation_importance_gru(i) for i, name in enumerate(FEAT_NAMES)}).sort_values(ascending=False)
print('Permutation importance (Brier degradation when shuffled -- higher = more important):')
print(perm_importance.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
perm_importance.plot.barh(ax=ax, color='#7c3aed')
ax.invert_yaxis()
ax.set_xlabel('Test Brier degradation when shuffled')
ax.set_title('Exp C (GRU) -- per-step feature importance (permutation)')
plt.tight_layout()
plt.savefig('../models/exp_c_feature_importance.png', dpi=120)
plt.show()

## Step 14 — Save artifacts

Model weights + all preprocessing metadata needed to reproduce inference (scaling constants,
`MAX_LEN`, calibration params) saved to `../models/exp_c_gru.pt` (gitignored — see `.gitignore`).
Combined 4-way test predictions saved alongside `test_predictions.parquet` for future reuse.

In [ ]:
torch.save({
    'model_state_dict': best_state,
    'hyperparams': {
        'hidden_size': HIDDEN_SIZE, 'dropout': DROPOUT, 'weight_decay': WEIGHT_DECAY,
        'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN, 'n_feats': N_FEATS,
        'random_state': RANDOM_STATE,
    },
    'scaling': {
        'setmargin_scale': SETMARGIN_SCALE, 'daysrest_clip': DAYSREST_CLIP, 'ligakoef_scale': LIGAKOEF_SCALE,
    },
    'feature_names': FEAT_NAMES + ['IsPlaceholder'],
    'calibration': {'method': calib_method, 'temperature': T_value},
    'best_eval_brier': float(best_brier),
    'test_metrics': compute_metrics(test_y_np, p_test_calibrated, 'Exp C - ELO-free GRU (calibrated)'),
}, MODELS_DIR / 'exp_c_gru.pt')

with open(MODELS_DIR / 'exp_c_isotonic.pkl', 'wb') as f:
    pickle.dump(iso, f)

exp_c_preds = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'PlayerId', 'OpponentId', 'Label', 'Perspective']].copy()
exp_c_preds['p_expC_raw'] = p_test_raw
exp_c_preds['p_expC_calibrated'] = p_test_calibrated
four_way = preds.merge(exp_c_preds[['SusretId', 'PlayerId', 'OpponentId', 'p_expC_raw', 'p_expC_calibrated']],
                        on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
assert len(four_way) == len(preds)
four_way.to_parquet(DATA_DIR / 'test_predictions_4way.parquet', index=False)

print(f"Saved: {MODELS_DIR / 'exp_c_gru.pt'}")
print(f"Saved: {MODELS_DIR / 'exp_c_isotonic.pkl'}")
print(f"Saved: {DATA_DIR / 'test_predictions_4way.parquet'}  (shape={four_way.shape})")

## Step 15 -- Robustness checks (adversarial audit, 2026-07-10)

An independent adversarial audit of this notebook confirmed the headline -8.24% Brier result is
real and reproducible (no leakage) but flagged two corrections, both addressed here with code that
actually retrains and measures -- not just asserted in prose.

**(a) Train/test hygiene -- `LIGAKOEF_SCALE`.** Step 5 originally hardcoded `LIGAKOEF_SCALE = 8.0`,
described there as "observed max LigaKoeficjent (see describe() above)" -- but that `describe()`
(Step 3 output) was computed over `player_log` before any train/test split existed, i.e. over the
FULL train+test frame. `LigaKoeficjent`'s max is only **7 within TRAIN**; **8** appears for the
first time only in the TEST period (a new razina level introduced after the split date) -- so the
old constant leaked a test-period fact into a train-only preprocessing decision. Step 5 above has
already been fixed to compute this from TRAIN ONLY. A GRU's first-layer weights can absorb any
fixed linear rescale of an input dimension, so this is expected to be numerically inert -- verified
below by retraining with the OLD (leaky) constant and comparing against the fixed model above.

**(b) What actually drives Exp C's edge over ELO?** The Written Verdict (Step 17 below), as
originally drafted, attributed the edge to "ELO only looks at win/loss, never margin-of-victory."
**That claim is retracted here.** Our production ELO update (implemented in the private club
app this data comes from, not part of this repo) is called with the full per-set scores for both
players on every match. The exact internal weighting lives in a compiled third-party library we
don't have source access to, so a flat "ELO never sees margin" statement cannot be made -- only
that it was wrong to assert it outright, and it's removed below. To find the real driver, five variants of the identical architecture / hyperparameters /
seed / data splits are retrained below, each with a different subset of the 6 real per-step
features zeroed out (a pure input-masking ablation, not a re-tuned model):

| Variant | Features kept (all others zeroed) |
|---|---|
| `FULL` | all 6 (reuses the already-trained model from Steps 8-11 above, incl. the Step 5 fix) |
| `NO_SETMARGIN` | drop `SetMarginScaled` |
| `LIGAKOEF_ONLY` | `Won` + `LigaKoefScaled` only |
| `NO_LIGAKOEF` | drop `LigaKoefScaled` |
| `WON_ONLY` | `Won` only |


In [ ]:
# --- (a) LIGAKOEF_SCALE train/test-hygiene fix: before/after comparison ---
# `run_variant` retrains the identical TwoTowerGRU architecture with a different LIGAKOEF_SCALE
# and/or a subset of the 6 real per-step features zeroed out. Reuses every already-built global
# from Steps 1-11 above (player_log, sym, idx_lookup, first_idx, PLACEHOLDER_VEC, MAX_LEN, N_FEATS,
# SETMARGIN_SCALE, DAYSREST_CLIP, igrac_arr, susret_arr, LEN_T, player_pos, opp_pos, HIDDEN_SIZE,
# DROPOUT, WEIGHT_DECAY, LR, BATCH_SIZE, MAX_EPOCHS, PATIENCE, TwoTowerGRU, compute_metrics, elo_brier).
LIGAKOEF_SCALE_OLD = 8.0  # the old, leaky constant (max computed over the FULL train+test frame)


def run_variant(ligakoef_scale, zero_idx, label, max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=RANDOM_STATE):
    sf = np.zeros((len(player_log), N_FEATS), dtype=np.float32)
    sf[:, 0] = player_log['Won'].values
    sf[:, 1] = np.clip(player_log['SetMargin'].values / SETMARGIN_SCALE, -3, 3)
    days_rest = player_log['DaysSinceLastMatch'].fillna(DAYSREST_CLIP).clip(upper=DAYSREST_CLIP).values
    sf[:, 2] = days_rest / DAYSREST_CLIP
    sf[:, 3] = player_log['LigaKoeficjent'].values / ligakoef_scale
    sf[:, 4] = player_log['HomeShareBefore'].fillna(0.0).values
    sf[:, 5] = player_log['OpponentCareerWinRateBefore'].fillna(0.5).values
    sf[:, 6] = 0.0
    for idx in zero_idx:
        sf[:, idx] = 0.0

    hist = np.zeros((len(player_log), MAX_LEN, N_FEATS), dtype=np.float32)
    for i in range(len(player_log)):
        gpos = idx_lookup[(igrac_arr[i], susret_arr[i])]
        start = first_idx[igrac_arr[i]]
        p = gpos - start
        if p == 0:
            hist[i, 0] = PLACEHOLDER_VEC
        else:
            lo = max(start, gpos - MAX_LEN)
            chunk = sf[lo:gpos]
            hist[i, :chunk.shape[0]] = chunk
    hist_t = torch.from_numpy(hist)

    def _bt(bucket_name):
        mask = (sym['Bucket'] == bucket_name).values
        pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
        op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
        y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
        return hist_t[pp], LEN_T[pp], hist_t[op], LEN_T[op], y

    v_core, v_eval, v_calib, v_test = _bt('core'), _bt('eval'), _bt('calib'), _bt('test')

    torch.manual_seed(seed)
    m = TwoTowerGRU(hidden_size=HIDDEN_SIZE, dropout=DROPOUT).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    crit = nn.BCEWithLogitsLoss()
    n_v_core = v_core[4].shape[0]
    rng_local = np.random.RandomState(seed)

    best_brier, best_state, no_improve = np.inf, None, 0
    for epoch in range(1, max_epochs + 1):
        m.train()
        perm = rng_local.permutation(n_v_core)
        for start in range(0, n_v_core, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            opt.zero_grad()
            logit = m(v_core[0][idx], v_core[1][idx], v_core[2][idx], v_core[3][idx])
            loss = crit(logit, v_core[4][idx])
            loss.backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            val_logit = m(v_eval[0], v_eval[1], v_eval[2], v_eval[3])
        val_brier = brier_score_loss(v_eval[4].numpy(), torch.sigmoid(val_logit).numpy())
        if val_brier < best_brier - 1e-5:
            best_brier, best_state, no_improve = val_brier, {k: v.clone() for k, v in m.state_dict().items()}, 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    m.load_state_dict(best_state)

    m.eval()
    with torch.no_grad():
        calib_logit = m(v_calib[0], v_calib[1], v_calib[2], v_calib[3])
        test_logit = m(v_test[0], v_test[1], v_test[2], v_test[3])
    calib_logit_np, calib_y_np = calib_logit.numpy(), v_calib[4].numpy()
    test_logit_np, test_y_np_v = test_logit.numpy(), v_test[4].numpy()

    T_p = torch.nn.Parameter(torch.ones(1))
    clt, cyt = torch.from_numpy(calib_logit_np), torch.from_numpy(calib_y_np)
    opt_T = torch.optim.LBFGS([T_p], lr=0.05, max_iter=100)

    def _closure():
        opt_T.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(clt / T_p, cyt)
        loss.backward()
        return loss
    opt_T.step(_closure)
    T_v = float(T_p.detach().clamp(min=1e-2))

    p_calib_raw_v = 1 / (1 + np.exp(-calib_logit_np))
    p_calib_temp_v = torch.sigmoid(clt / T_v).numpy()
    iso_v = IsotonicRegression(out_of_bounds='clip')
    iso_v.fit(p_calib_raw_v, calib_y_np)
    p_calib_iso_v = iso_v.predict(p_calib_raw_v)

    p_test_raw_v = 1 / (1 + np.exp(-test_logit_np))
    if brier_score_loss(calib_y_np, p_calib_iso_v) < brier_score_loss(calib_y_np, p_calib_temp_v):
        p_test_cal_v = iso_v.predict(p_test_raw_v)
    else:
        p_test_cal_v = torch.sigmoid(torch.from_numpy(test_logit_np) / T_v).numpy()

    return compute_metrics(test_y_np_v, p_test_cal_v, label)


before_fix = run_variant(LIGAKOEF_SCALE_OLD, [], 'FULL (OLD scale=8, pre-fix repro)')
after_fix = compute_metrics(test_y_np, p_test_calibrated, 'FULL (train-only scale=7, this notebook)')
scale_check = pd.DataFrame([before_fix, after_fix]).set_index('experiment')
scale_check['brier_delta_vs_elo_pct'] = (scale_check['brier'] - elo_brier) / elo_brier * 100
print(f"Brier delta (after-fix - before-fix): {after_fix['brier'] - before_fix['brier']:+.5f}")
scale_check.round({'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})

In [ ]:
# --- (b) Ablation: which of the 6 real per-step features actually drives the edge? ---
ABLATION_VARIANTS = [
    ('FULL', []),
    ('NO_SETMARGIN', [1]),
    ('LIGAKOEF_ONLY', [1, 2, 4, 5]),
    ('NO_LIGAKOEF', [3]),
    ('WON_ONLY', [1, 2, 3, 4, 5]),
]

ablation_rows = []
for label, zero_idx in ABLATION_VARIANTS:
    if label == 'FULL':
        row = compute_metrics(test_y_np, p_test_calibrated, label)  # reuse the already-trained main model (Steps 8-11)
    else:
        row = run_variant(LIGAKOEF_SCALE, zero_idx, label)
    row['brier_delta_vs_elo_pct'] = (row['brier'] - elo_brier) / elo_brier * 100
    ablation_rows.append(row)

ablation_table = pd.DataFrame(ablation_rows).set_index('experiment')
ablation_table_display = ablation_table.round(
    {'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})
ablation_table_display

## Step 16 — Seed-stability check (reported, not retrained inline — see note)

NN training carries residual stochasticity even with a fixed seed (weight init, shuffle order,
dropout masks). This exact pipeline (identical architecture/hyperparams/data splits) was run
end-to-end 3 times externally with seeds `{42, 123, 7}` before finalizing this notebook:

| Seed | Test Brier (calibrated) | Test Accuracy | Test AUC |
|---|---|---|---|
| 42, OLD scale=8 (pre-fix repro, see Step 15) | 0.16254 | 0.7633 | 0.8417 |
| 42, train-only scale=7 (this notebook, post-fix) | 0.16278 | 0.7621 | 0.8411 |
| 123, OLD scale=8 | 0.15976 | 0.7640 | 0.8466 |
| 7, OLD scale=8 | 0.15921 | 0.7640 | 0.8478 |

Mean Brier (3 original seeds, OLD scale) 0.1605, std 0.0018 -- all three beat Exp B (0.16699) and
Exp 0 (0.17713) by a comfortable margin, well outside the ~0.002 run-to-run noise band. The Step 15
train-only-scale fix moves seed 42 by +0.00024 -- more than an order of magnitude smaller than this
noise band, i.e. clearly inert. This is a stable finding, not a lucky seed (and not an artifact of
the since-fixed scale constant either).

## Step 17 -- Written Verdict

**Did the GRU beat Exp A (ELO-free XGBoost)? Emphatically yes.** Exp C's calibrated test Brier
(~0.162-0.163, seed 42; 0.159-0.163 across 3 seeds, see Step 16) is dramatically better than Exp
A's 0.19690 -- an ~18% relative Brier reduction. Both experiments are ELO-free; the difference is
entirely in HOW the ELO-free signal is consumed: Exp A compresses each player's history into ~10
static summary statistics (career win rate, rolling forma, H2H rate, razina/debut placement,
clutch rates) fed to a tree ensemble; Exp C feeds the RAW sequence of individual match outcomes
(with per-match set margin, rest, razina, and opponent-quality-at-the-time) to a recurrent network
that learns its own compression.

**Did it approach or beat Exp 0 (ELO baseline, Brier 0.1771/acc 75.18%) and Exp B (ELO-included
XGBoost, Brier 0.1670)? On THIS single 80/20 split, yes, by a comfortable point-estimate margin,
and it did so WITHOUT ever seeing an ELO number.** Test Brier ~0.162-0.163 vs ELO's 0.1771 (~8%
relative improvement) and vs Exp B's 0.1670 (~3% relative improvement over the strongest
ELO-aware tabular model built at the time this notebook was written). This directly contradicts
the expectation set going in ("dramatically beating ELO is unlikely, data-limited; a null/tie is a
legitimate finding") -- the honest, falsifiable thesis question ("can ML beat ELO without
ELO?") gets a clear **yes** here, on this test set.

**This single-split "beats Exp B" claim is the part that did NOT hold up under further scrutiny --
flagged here, not buried.** `06_confidence_intervals_cv.ipynb`'s paired bootstrap (2,000
match-level resamples) found the (Exp C - Exp B) Brier gap's 95% CI straddles 0 -- not
statistically significant, despite the point estimate favoring Exp C. Its walk-forward CV (4
expanding time folds) found Exp C beats Exp B in only 1 of 4 folds, with Exp B ahead on average
across folds. The "beats ELO (Exp 0)" half of this claim held up much better: significant on the
paired bootstrap, and directionally correct in 3 of 4 walk-forward folds (losing only in the
earliest, thinnest-data fold -- a cold-start effect, not noise). See `06` for the full breakdown.

**Corrected mechanism (adversarial audit, 2026-07-10) -- this is the section that changed.** The
original draft of this verdict claimed the edge came from margin-of-victory, on the premise that
"ELO's update only looks at win/loss and ignores margin." **That premise is wrong and has been
removed.** Our production ELO update (implemented in the private club app this data comes from, not
part of this repo) is called with the full per-set scores for both players on every match -- so a
flat "ELO never sees margin" statement is false; the exact internal weighting lives in a compiled
third-party library we don't have source access to, so no claim is made either way about how much
margin influences the production ELO number.

The Step 15 ablation (five retrained variants, each with a different subset of the 6 real per-step
features zeroed out) identifies the REAL driver:

| Variant | Features kept | Test Brier | vs ELO |
|---|---|---|---|
| FULL | all 6 per-step features | ~0.162-0.163 | ~-8% |
| NO_SETMARGIN | drop SetMargin | ~0.167 | ~-5.5% |
| LIGAKOEF_ONLY | Won + LigaKoef only | ~0.167 | ~-5.5% |
| NO_LIGAKOEF | drop LigaKoef | ~0.183 | ~+3.5% (WORSE than ELO) |
| WON_ONLY | Won only | ~0.216 | ~+22% |

(Exact numbers from this run's Step 15 table -- restated here at reduced precision because a GRU
has no deterministic CPU kernel path, see Step 1's `torch.use_deterministic_algorithms(False)`
note, so re-running shifts the 4th-5th decimal; the ranking and magnitude gaps between variants are
the stable, reportable finding, not any single decimal.)

**Correct interpretation: feeding a player's razina (LigaKoeficjent tier-placement) *trajectory*
as a sequence to the GRU extracts more signal than either the numeric ELO rating (Exp 0) or a
single static razina scalar fed to XGBoost (Exp A/B).** Dropping `LigaKoefScaled` alone
(`NO_LIGAKOEF`) is the only ablation that makes the GRU WORSE than the ELO baseline it's supposed
to beat -- razina is doing essentially all of the heavy lifting. `LIGAKOEF_ONLY` (just `Won` +
razina, nothing else) already recovers almost the entire FULL-model edge over ELO. `SetMargin` adds
a genuine but secondary lift on top of razina (`FULL` vs `NO_SETMARGIN` narrows the gap vs ELO from
~5.5% to ~8%) -- real, but clearly not the primary mechanism the original verdict claimed.
`DaysSinceLastMatch` / `HomeShareBefore` / `OpponentCareerWinRateBefore` are not isolated
separately above (all already off in every non-FULL row except FULL itself), but their
contribution is implicitly small: `NO_SETMARGIN` (drops only 1 of 5 remaining non-razina/non-Won
features) already nearly matches `LIGAKOEF_ONLY` (drops all 4), meaning DaysSinceLastMatch /
HomeShareBefore / OpponentCareerWinRateBefore collectively add almost nothing beyond what razina +
Won already capture.

Razina (an admin/human judgment -- promotion, relegation, wildcard placement) is correlated with
skill (`corr(LigaKoeficjent, EloBefore) ~ -0.66` in this dataset) but is NOT simply a discretized
ELO: within-tier ELO standard deviation ranges from ~125 to ~305 rating points depending on the
tier (see the correlation check run during this audit), meaning two players in the same razina can
be materially different in ELO. So the effect is a legitimate structural signal, not a circular
"the GRU rediscovered ELO through the back door" artifact -- the GRU is reading something ELO
itself does not directly encode: WHERE administrators have placed a player, and how that placement
has evolved over their career.

**Honest framing:** this is a cleaner, more interesting, TRUE finding than the original draft's --
"a sequence of administrative tier-placement decisions beats a numeric ELO rating, for this
dataset" -- not "the GRU learned a better rating from raw results + margin." It is also a more
falsifiable and mechanistically grounded claim: administrators changing a player's razina is a
real-world signal (promotions/relegations/wildcards reflecting human judgment of current form)
that a static ELO number, updated only match-by-match by a fixed formula, cannot fully capture as
quickly.

**Calibration:** both temperature scaling and isotonic regression were fit on the same held-out
calibration slice; isotonic won in this run (see Step 10's calib-slice Brier comparison) and was
applied to TEST. The calibration curve (Step 12) is visually competitive with Exp B's and clearly
tighter than Exp A's mid-range wobble -- read against the diagonal in the saved
`calibration_curve_4way.png`.

**Train/test hygiene (adversarial audit, 2026-07-10):** `LIGAKOEF_SCALE` was originally computed
(informally) over the full train+test frame's `describe()` (max=8), when the TRAIN-only max is 7 --
a new razina level (8) appears for the first time only in TEST. Step 5 now computes this from TRAIN
ONLY. Step 15(a) verifies the fix is numerically inert (before/after Brier delta is a small
fraction of the ~0.0018 seed-to-seed noise band established in Step 16) -- exactly as expected,
since the GRU's first-layer weights can absorb any fixed linear rescale of an input dimension.

**Honest caveats (this is still a small, single-club dataset):**
- ~17k matches, ~34k duplicate-and-flip rows, one 80/20 time split (no walk-forward
  re-validation across multiple season boundaries) -- same scope limits flagged in `04_model_evaluation.ipynb`.
- The opponent-quality proxy (`OpponentCareerWinRateBefore`) is a single scalar per step, not a
  recursive opponent-history embedding -- a genuinely richer (and out-of-scope for this
  notebook) architecture might do even better, or might not: it's untested.
- The ablation zeroes features rather than dropping the input dimension entirely (retraining with
  a smaller `input_size`) -- a standard, valid masking technique, but not identical to a
  from-scratch smaller-architecture retrain; the qualitative ranking (razina >> everything else,
  SetMargin secondary, rest/home/opponent-quality negligible) is the reportable finding, not the
  exact decimal deltas.
- 3-seed stability check (Step 16) is reassuring but not a full multi-seed distribution; a
  production decision would want more runs and a walk-forward validation across additional
  season boundaries before fully trusting the margin over Exp B.
- Despite the strong test-set result, this is still a recreational, single-club, ~9-year dataset
  -- the finding that razina-trajectory beats a numeric ELO rating is plausible and mechanistically
  explainable here, but the specific magnitude of the improvement should be read as a promising,
  well-verified directional result on this dataset, not a universal claim about GRUs beating ELO in
  general.

**Bottom line (as this notebook stood on 2026-07-10):** this notebook set out to give an honest
report either way -- this is the "either way" that happened to be the more exciting outcome on this test set,
now attributed to the correct mechanism (razina-trajectory, not margin-of-victory). Exp C (ELO-free
GRU) beats every other model built so far in this notebook, including the ELO-included XGBoost,
while never being told the ELO number. **This is a real, single-split result, not a mistake -- but
it is not the final word on "is Exp C the best model," and the notebooks that follow are what
actually settle that question. Treat everything below this line as historical: it reflects what
was known when this notebook was written, before the follow-up work existed.**

**Where this stands today (added after `06`-`09` were run):**
- `06_confidence_intervals_cv.ipynb` -- bootstrap + walk-forward CV on this exact Exp 0/A/B/C set.
  Exp C beating Exp 0 (ELO) is statistically significant and holds in 3/4 time folds (cold-start
  miss in the thinnest fold). Exp C beating Exp B is NOT statistically significant and flips to an
  Exp B win in 3/4 time folds -- the "beats everything" framing above does not survive this check.
- `07_xgboost_trajectory.ipynb` -- traces Exp C's edge to its mechanism (confirmed here in Step
  15/16: razina-trajectory). When XGBoost is handed the same razina trajectory as 9 hand-engineered
  features (Exp A'), it closes roughly two-thirds of the ELO-free gap to Exp C and, for the first
  time, an ELO-free tabular model edges past plain ELO. A real, smaller, statistically significant
  remainder still favors the GRU -- so the razina-trajectory MECHANISM identified here holds up, but
  it turns out to be mostly an input-representation advantage (trajectory vs. static snapshot), not
  a synergy unique to recurrent architectures.
- `08_gru_elo.ipynb` -- adds point-in-time ELO to this same GRU (Exp C2). Point estimate improves
  further (best Brier in the whole project) and *is* now significantly better than every XGBoost
  variant -- but the gain over the ELO-free Exp C itself is NOT statistically significant. ELO and
  the GRU's own learned representation turn out to be close substitutes, not complements.
- `09_walkforward_top_models.ipynb` (capstone) -- walk-forward tests the two strongest successors
  (Exp B', Exp C2) instead of the original four. Exp C2 beats plain ELO in 4/4 folds (robust, unlike
  Exp C's 3/4 here) and beats Exp B' in 3/4 folds -- but loses to Exp B' in the earliest,
  thinnest-data fold. The mature, walk-forward-validated conclusion is: **Exp C2 is the best
  available model on the current full dataset, but ELO is what makes the GRU robust in low-data
  regimes, and a well-tuned ELO-aware XGBoost model (Exp B') remains competitive-to-better when
  training history is thin.** See `09`'s Written Summary and `../README.md` for the full arc.

**None of this erases the finding actually established in this notebook** -- the razina-trajectory
mechanism (Step 15/16 ablation) is real, audit-confirmed, and remains the correct explanation for
why an ELO-free sequence model can compete with ELO at all. What changed is the conclusion drawn
from it: not "Exp C is the best model, full stop," but "razina-trajectory is a genuine, ELO-free
signal, and how much that buys you (over XGBoost, over ELO, over the GRU-plus-ELO variant) depends
on which comparison and which time period you look at" -- which is exactly what the rest of this
project (`06`-`09`) goes on to quantify.
